In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3
!pip -q install speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3
!pip -q install "einops>=0.8.0" 2>&1 | tail -3


In [ ]:
import hashlib
import time as _clock
NOTEBOOK_START = _clock.time()
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/omnivoice_adapt.py") if "persian-voice-longcat-probe" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/omnivoice_adapt.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-longcat-probe-run-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def fetch(plan, minutes, flags=()):
    """scripts/fetch_inputs.py under run_bounded: every request timed out, the stage bounded."""
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text(json.dumps(plan))
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *flags], timeout=minutes * 60, log=LOGS / "fetch.log",
                      monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    return run.ok and report.get("ok") is True, fetched

os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable
os.environ["VOICECLONE_LONGCAT_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
NEEDED = ["scripts/prepare_probe_corpus.py", "scripts/read_longcat_probe.py", "scripts/longcat_worker.py",
          "eval/manifests/probe_texts_en_v1.json", "src/voiceclone/engines/longcat_tts.py"]
missing = [name for name in NEEDED if not (REPO / name).is_file()]
GATES.record("code", not missing, f"missing from the dataset: {missing}" if missing else "the probe's code is present")
free = shutil.disk_usage("/tmp").free / 2**30
if not GATES.skip("download", ("runner", "code")):
    if free < 40:
        GATES.record("download", False, f"{free:.0f} GiB free under /tmp, 40 needed")
    else:
        fetched_ok, fetched = fetch({'urls': [{'name': 'longcat code audiodit/__init__.py', 'url': 'https://raw.githubusercontent.com/meituan-longcat/LongCat-AudioDiT/12c76b51d2a8aa6b6c9af5b25cd5ff8f7aa8178a/audiodit/__init__.py', 'path': '/tmp/longcat-code/audiodit/__init__.py', 'git_blob_prefix': '3e02bb188c3729e3cb18f45226e73a7d43ec6a1f'}, {'name': 'longcat code audiodit/configuration_audiodit.py', 'url': 'https://raw.githubusercontent.com/meituan-longcat/LongCat-AudioDiT/12c76b51d2a8aa6b6c9af5b25cd5ff8f7aa8178a/audiodit/configuration_audiodit.py', 'path': '/tmp/longcat-code/audiodit/configuration_audiodit.py', 'git_blob_prefix': '36f7aacb0d535fd9bf83d7c251626544a3e17ac0'}, {'name': 'longcat code audiodit/modeling_audiodit.py', 'url': 'https://raw.githubusercontent.com/meituan-longcat/LongCat-AudioDiT/12c76b51d2a8aa6b6c9af5b25cd5ff8f7aa8178a/audiodit/modeling_audiodit.py', 'path': '/tmp/longcat-code/audiodit/modeling_audiodit.py', 'git_blob_prefix': '1333cc79ee449adbebe767772fe8fd16bb2d9c9f'}, {'name': 'longcat code utils.py', 'url': 'https://raw.githubusercontent.com/meituan-longcat/LongCat-AudioDiT/12c76b51d2a8aa6b6c9af5b25cd5ff8f7aa8178a/utils.py', 'path': '/tmp/longcat-code/utils.py', 'git_blob_prefix': '1e208da4d160de40f90adfeff5d82149618ac2b0'}], 'snapshots': [{'name': 'omnivoice base', 'repo': 'k2-fsa/OmniVoice', 'revision': 'c5fdb5ccb189668d56333f77ba2629f4cd7535f4', 'local_dir': '/tmp/omnivoice-base', 'sha256': {'audio_tokenizer/model.safetensors': 'fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d'}}, {'name': 'longcat 1b', 'repo': 'meituan-longcat/LongCat-AudioDiT-1B', 'revision': '4f4c19434f2e3b991998d0ead6ab2cd056d947b7', 'allow_patterns': ['config.json', 'model.safetensors', 'LICENSE'], 'local_dir': '/tmp/longcat-1b', 'sha256': {'model.safetensors': '7f41b20933e4466400b8487fd20ca195efa65c5ca7c61f8e9bba6316aa3edcde'}}, {'name': 'longcat 3.5b', 'repo': 'meituan-longcat/LongCat-AudioDiT-3.5B', 'revision': '5d3e6dc007872d756c3a69b44951598d7d9bdf6d', 'allow_patterns': ['config.json', 'model.safetensors', 'LICENSE'], 'local_dir': '/tmp/longcat-3.5b', 'sha256': {'model.safetensors': '708cdbf0bdf71cd485bf10c70a6043b22729730ef55a6b4bc2de2ce552e52fd8'}}, {'name': 'umt5 tokenizer', 'repo': 'google/umt5-base', 'revision': '0de9394d54f8975e71838d309de1cb496c894ab9', 'allow_patterns': ['spiece.model', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'config.json'], 'local_dir': '/tmp/umt5-base', 'sha256': {'spiece.model': 'e3909a67b780650b35cf529ac782ad2b6b26e6d1f849d3fbb6a872905f452458', 'tokenizer.json': 'af904105ce1071b1202bba0059a841f4a7b85b48b6ec179c4948e3483476e0dd'}}, {'name': 'neyshekar valid', 'repo': 'shekar-ai/neyshekar', 'repo_type': 'dataset', 'revision': '30696ada746a66bb3047c984744b5ecfd60fe4e9', 'allow_patterns': ['data/validation-*'], 'local_dir': '/tmp/neyshekar-valid', 'forbid': ['train-*', 'test-*']}]}, 45, ['--no-xet'])
        GATES.record("download", fetched_ok, json.dumps(fetched))


In [ ]:
CORPUS = Path("/tmp/probe-ney")
SPEAKERS = 0
if not GATES.skip("corpus", ("download",)):
    run = run_bounded([sys.executable, "scripts/prepare_probe_corpus.py", "--splits", str(SPLITS),
                       "--parquet-dir", "/tmp/neyshekar-valid", "--out", str(CORPUS)],
                      cwd=REPO, timeout=1200, log=LOGS / "corpus.log")
    RUNS["corpus"] = run.record()
    print(run.summary())
    roster = json.loads((CORPUS / "roster.json").read_text()) if (CORPUS / "roster.json").is_file() else {}
    SPEAKERS = len(roster.get("speakers", []))
    GATES.record("corpus", run.ok and SPEAKERS >= 20,
                 f"{SPEAKERS} speakers; left out: {roster.get('left_out')}")


In [ ]:
if not GATES.skip("configure", ("corpus",)):
    os.environ["VOICECLONE__ENGINES__LONGCAT__CODE_DIR"] = "/tmp/longcat-code"
    os.environ["VOICECLONE__ENGINES__LONGCAT__TOKENIZER_DIR"] = "/tmp/umt5-base"
    present = {path: Path(path).exists() for path in (
        "/tmp/omnivoice-base/config.json", "/tmp/longcat-1b/model.safetensors", "/tmp/longcat-3.5b/model.safetensors",
        "/tmp/longcat-code/audiodit/modeling_audiodit.py", "/tmp/longcat-code/utils.py", "/tmp/umt5-base/spiece.model")}
    GATES.record("configure", all(present.values()), json.dumps(present))


In [ ]:
ARMS = [['omnivoice-probe-en', 'omnivoice', {'VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID': '/tmp/omnivoice-base', 'VOICECLONE__ENGINES__ASR__LANGUAGE': 'en'}, ['--sentences-file', 'eval/manifests/probe_texts_en_v1.json', '--engine-option', 'language=en'], 6, 0.85, 'primary'], ['longcat-1b-probe-en', 'longcat', {'VOICECLONE__ENGINES__LONGCAT__MODEL_DIR': '/tmp/longcat-1b', 'VOICECLONE__ENGINES__ASR__LANGUAGE': 'en'}, ['--sentences-file', 'eval/manifests/probe_texts_en_v1.json'], 6, 0.85, 'primary'], ['longcat-3.5b-probe-en', 'longcat', {'VOICECLONE__ENGINES__LONGCAT__MODEL_DIR': '/tmp/longcat-3.5b', 'VOICECLONE__ENGINES__LONGCAT__TRANSFORMER_DEVICE': 'cuda:1', 'VOICECLONE__ENGINES__ASR__LANGUAGE': 'en'}, ['--sentences-file', 'eval/manifests/probe_texts_en_v1.json'], 6, 1.2, 'secondary'], ['omnivoice-probe-fa', 'omnivoice', {'VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID': '/tmp/omnivoice-base', 'VOICECLONE__ENGINES__ASR__LANGUAGE': 'fa'}, ['--sentences', 's1', 's2'], 2, 0.35, 'persian'], ['longcat-1b-probe-fa', 'longcat', {'VOICECLONE__ENGINES__LONGCAT__MODEL_DIR': '/tmp/longcat-1b', 'VOICECLONE__ENGINES__ASR__LANGUAGE': 'fa'}, ['--sentences', 's1', 's2'], 2, 0.35, 'persian']]
FACTS = {}


def remaining_hours():
    return 2.0 - (time.time() - NOTEBOOK_START) / 3600


def bench(label, engine, env, extra, per_speaker, hours):
    left = remaining_hours() - 0.1
    if left < 0.15:
        print(f"{label}: {remaining_hours():.2f} h left in the session - not started")
        return False, {}
    saved = {key: os.environ.get(key) for key in env}
    os.environ.update(env)
    try:
        run = run_bounded([sys.executable, "scripts/benchmark.py", "--label", label, "--engine", engine,
                           "--corpus-dir", str(CORPUS), "--seeds", "42", "--device", "cuda", *extra],
                          cwd=REPO, timeout=min(hours, left) * 3600, log=LOGS / f"{label}.log", monitor_every=600)
    finally:
        for key, value in saved.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value
    RUNS[label] = run.record()
    print(run.summary())
    results = REPO / "eval/results" / f"{label}.json"
    summary = json.loads(results.read_text())["summary"] if results.is_file() else {}
    return run.ok and summary.get("renders", 0) >= 0.9 * SPEAKERS * per_speaker, summary


if not GATES.skip("renders", ("configure",)):
    for label, engine, env, extra, per_speaker, hours, role in ARMS:
        passed, summary = bench(label, engine, env, extra, per_speaker, hours)
        FACTS[label] = {"role": role, "complete": passed, "held_out": summary.get("held_out_mean"),
                        "reference": summary.get("reference_mean"), "cer": summary.get("cer_mean"),
                        "renders": summary.get("renders"), "failures": summary.get("failures")}
        print(label, json.dumps(FACTS[label]))
    for role in ("primary", "secondary", "persian"):
        rows = {label: facts for label, facts in FACTS.items() if facts["role"] == role}
        GATES.record(role, bool(rows) and all(f["complete"] for f in rows.values()), json.dumps(rows))


In [ ]:
READING = {}
if not GATES.skip("reading", ("primary",)):
    have = {label for label, *_ in ARMS if (REPO / "eval/results" / f"{label}.json").is_file()}
    arguments = ["--omnivoice", "omnivoice-probe-en", "--longcat", "longcat-1b-probe-en",
                 "--out", str(RECORDS / "reading.json")]
    if "longcat-3.5b-probe-en" in have:
        arguments += ["--secondary", "longcat-3.5b-probe-en"]
    if {"omnivoice-probe-fa", "longcat-1b-probe-fa"} <= have:
        arguments += ["--omnivoice-fa", "omnivoice-probe-fa", "--longcat-fa", "longcat-1b-probe-fa"]
    run = run_bounded([sys.executable, "scripts/read_longcat_probe.py", *arguments], cwd=REPO, timeout=600,
                      log=LOGS / "reading.log")
    RUNS["reading"] = run.record()
    READING = json.loads((RECORDS / "reading.json").read_text()) if (RECORDS / "reading.json").is_file() else {}
    GATES.record("reading", run.ok and bool(READING), json.dumps(READING.get("decision")))
    print(json.dumps(READING.get("primary", {}).get("verdict")))


In [ ]:
destination = Path("/kaggle/working/longcat-probe")
destination.mkdir(exist_ok=True)
for label, *_ in ARMS:
    for suffix in (".json", ".recipe.json"):
        path = REPO / "eval/results" / f"{label}{suffix}"
        if path.is_file():
            shutil.copy2(path, destination / path.name)
for name in ("roster.json", "manifest.json"):
    if (CORPUS / name).is_file():
        shutil.copy2(CORPUS / name, destination / f"corpus-{name}")
for path in RECORDS.glob("*.json"):
    shutil.copy2(path, destination / path.name)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
(destination / "facts.json").write_text(json.dumps(FACTS, indent=2))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
renders = Path("/kaggle/working/renders")
renders.mkdir(exist_ok=True)
for label, *_ in ARMS:
    folder = REPO / "outputs/benchmark" / label
    for path in (folder.glob("*.wav") if folder.is_dir() else []):
        shutil.copy2(path, renders / f"{label}__{path.name}")
print("gates:", json.dumps(GATES.summary(), indent=1))
